In [2]:
from sql.game import Game
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [3]:
from sql.session import session
from sqlalchemy import select

results = session.execute(select(Game)).scalars().all()


df = pd.DataFrame([g.__dict__ for g in results]).set_index("name")

df = df.dropna(subset=["rank_all"])

In [4]:
def keep(col: str) -> bool:
    if "_sa_" in col:
        return False
    if col in {"thumbnail", "image", "description", "url", "wanting", "id", "trading", "owned", "wishing", "usersrated"}:
        return False
    if "num" in col:
        return False
    if "rank" in col:
        return False
    if col in ["min_play_time", "max_play_time", "min_players", "max_players"]: #colinearity with playing_time
        return False
    if col in ["average", "median", "stddev", "const"]:
        return False
    return True

columns_to_keep = [col for col in df if keep(col)]
df = df[df["usersrated"] > 1000]
df = df[df["year_published"] >= 1950]
df = df[columns_to_keep]
df = df[df["bayesaverage"] > 0.1]
df = df[df["averageweight"] > 0.0]
print(columns_to_keep, set(df) - set(columns_to_keep))

['playing_time', 'averageweight', 'year_published', 'min_age', 'best_player_count', 'language_dependence', 'bayesaverage', 'recommended_player_counts'] set()


In [5]:
from exploration.utils import add_player_group_features, add_best_player_count_features, decode_language_dependence

df = add_player_group_features(df, "recommended_player_counts")
df = df[df["recommended_for_valid"]==1].drop(columns=["recommended_for_valid", "recommended_player_counts", "best_player_count"])

In [6]:
# df = add_best_player_count_features(df, "best_player_count")
# df = df[df["best_player_count_valid"]==1].drop(columns=["best_player_count_valid", "best_player_count"])

In [7]:
df.language_dependence = df.language_dependence.apply(decode_language_dependence)

In [8]:
# Drop remaining missing values
df = df.dropna()

X = df[[col for col in df if "rating" not in col and "rank" not in col]]
X = (X - X.min()) / (X.max() - X.min()).replace(0, 1)

y = X.pop("bayesaverage") * 0.1

X = sm.add_constant(X)

In [11]:
import matplotlib.pyplot as plt

EARTH = {
    "line": "#8B5E3C",        # warm brown
    "density": "#6B8E23",     # olive green
    "accent": "#A3B18A",      # muted sage
    "zero": "#444444"
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#444444",
    "xtick.color": "#444444",
    "ytick.color": "#444444",
    "font.size": 11,
})